In [1]:
# add edw vitals
# add edw oxygen supply if not already there
# make sure correct (and only correct) sleep stage models and results are in data
# apnea pred. in data



In [58]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/AirGo_Apnea_Detection')
from apnea_detection_functions import apply_apnea_prediction_models

sys.path.append('/home/wolfgang/repos/ICU-Sleep/code1')
from airgo_features import compute_airgo_features
from sleep_staging_functions import apply_airgo_sleep_staging_models

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import *
from sleep_analysis_functions import *
from edw_functions import * 

%matplotlib widget
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
cohort = 'icu'
data_dir = f'/media/ssd2/{cohort}_final_files'
files = os.listdir(data_dir)
files.sort()
len(files)


LIST = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/LIST.csv'
LIST = pd.read_csv(LIST)
LIST.dropna(subset=['MRN:'], inplace=True)
LIST['study_id'] = LIST['Study ID:'].apply(lambda x: str(x).zfill(3))
LIST['mrn'] = LIST['MRN:'].apply(lambda x: int(x))


file =  files[48]
study_id = file[-6:-3]
print(file)


icusleep_081.h5


In [108]:
data, hdr, fs = read_in_routine(os.path.join(data_dir, file))

Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled


In [110]:
data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=True)

In [120]:
bm_signals = ['hr', 'spo2', 'nbps', 'nbpd', 'art1d', 'art1s', 'art2d', 'art2s']

original_airgo_signals = ['band', 'accx', 'accy', 'accz']

signals_to_keep = bm_signals + original_airgo_signals
signals_to_keep = [x for x in signals_to_keep if x in data.columns]


In [121]:
data = data[signals_to_keep]

In [129]:
data

,hr,spo2,art1d,art1s,band,accx,accy,accz
2019-05-11 22:09:23.000,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-11 22:09:23.100,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-11 22:09:23.200,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-11 22:09:23.300,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-11 22:09:23.400,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2019-05-15 11:52:02.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-15 11:52:02.700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-15 11:52:02.800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-05-15 11:52:02.900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
edw_oxygen = get_edw_oxygen_single_mrn(oxygen_path, mrn)

In [91]:
edw_oxygen

,ed_pre-arrival_o2_device,ed_pre-arrival_oxygen_flow_rate,oxygen_device,oxygen_flow_rate,phs_amb_pt_oxygen_flow_rate,positioning_frequency,repositioned
datetime,,,,,,,


In [95]:
data.band.dropna()

2019-05-14 19:41:48.700    2752.0
2019-05-14 19:41:48.800    2796.0
2019-05-14 19:41:50.100    2684.0
2019-05-14 19:41:50.200    2704.0
2019-05-14 19:41:50.300    2688.0
                            ...  
2019-05-15 11:48:19.300    2468.0
2019-05-15 11:48:19.400    2362.0
2019-05-15 11:48:19.500    2362.0
2019-05-15 11:48:19.600    2340.0
2019-05-15 11:48:19.700    2344.0
Name: band, Length: 579841, dtype: float32

In [ ]:
fs_manual = 10
do_resample_and_interpolation = False       # recommended for raw airgo, resampling to 'perfect 10Hz'
do_compute_airgo_features = False           # all features. by default, complexity features are computed in this code which are the slowest but needed for apnea prediction.
do_apply_sleep_staging_models = True        # respiration-only, actigraphy-only.
do_apply_apnea_prediction_models = True    # respiration-only and respiration+spo2 models. if sleep_stage_available, sleep-only apnea versions get computed too.
do_compute_self_similarity = True          # depends on airgo available
do_compute_hypoxia_burden = True           # depends on apnea predictions and sleep stages (for hours of sleep)


### compute_hypoxia_burden params ###
apnea_name = 'apnea_pred_va_a3'                      # name of Apnea info column
hours_sleep = 'stage_pred_amendsumeffort'  # name of sleep stage column, or int/float for manual setting.

